# Configuração do ambiente

In [367]:
from pyspark.sql import SparkSession

#Create SparkSession
spark = SparkSession.builder.getOrCreate()

In [368]:
from pyspark.sql import functions as F
import xgboost

In [369]:
import numpy as np

In [370]:
from  pyspark.ml import Pipeline

In [371]:
from pyspark.ml.feature import StringIndexer,VectorAssembler, OneHotEncoder
from pyspark.sql.types import  BooleanType, DateType, DoubleType, IntegerType, StringType

In [372]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from xgboost.spark import SparkXGBClassifier 


# Leitura dos dados

In [373]:
dados = (spark.read
         .format('csv')
         .option('header', 'true')  
         .option('sep' , ',')
         
         .option('inferSchema', 'true')  # Para inferir os tipos de dados automaticamente
         .load('C:/Users/Gabriel/Desktop/Projeto_spark/hotel_dataframe.csv')
        )


In [374]:
train, test = dados.randomSplit([0.8, 0.2], seed= 13)

In [375]:
train.printSchema()

root
 |-- no_of_adults: integer (nullable = true)
 |-- no_of_children: integer (nullable = true)
 |-- no_of_weekend_nights: integer (nullable = true)
 |-- no_of_week_nights: integer (nullable = true)
 |-- type_of_meal_plan: string (nullable = true)
 |-- required_car_parking_space: integer (nullable = true)
 |-- room_type_reserved: string (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- arrival_year: integer (nullable = true)
 |-- arrival_month: integer (nullable = true)
 |-- arrival_date: integer (nullable = true)
 |-- market_segment_type: string (nullable = true)
 |-- repeated_guest: integer (nullable = true)
 |-- no_of_previous_cancellations: integer (nullable = true)
 |-- no_of_previous_bookings_not_canceled: integer (nullable = true)
 |-- avg_price_per_room: double (nullable = true)
 |-- no_of_special_requests: integer (nullable = true)
 |-- booking_status: string (nullable = true)
 |-- is_duplicated: boolean (nullable = true)
 |-- duplicated_count: integer (nullabl

In [376]:
set([train.schema[i].dataType for i in range(len(train.columns))])

{BooleanType(), DateType(), DoubleType(), IntegerType(), StringType()}

# Pipeline 

In [392]:
#Remover target e atributos relacionados a data da matriz de atribuos
cols_to_use = train.columns
cols_to_use.remove('booking_status')
cols_to_use.remove('data') # Remover data (deve ser usada so para fins de validação/exploratório)
cols_to_use.remove('arrival_date') 

In [393]:
cat_cols_df = [col.name for col in train.schema if col.dataType == StringType()]

train.select(*[F.countDistinct(col).alias(col) for col in cat_cols_df]).show()





+-----------------+------------------+-------------------+--------------+---------+
|type_of_meal_plan|room_type_reserved|market_segment_type|booking_status|Trimestre|
+-----------------+------------------+-------------------+--------------+---------+
|                3|                 7|                  5|             2|        3|
+-----------------+------------------+-------------------+--------------+---------+



Como as features categoricas não tem cardinlidade alta, seguiremos com one hot encoding

In [394]:
stages = []

In [395]:
label_transform  = StringIndexer(inputCol="booking_status", outputCol="label")
stages.append(label_transform)

In [396]:
# colunas categoricas do pipeline , isso é que são codificadas como StringType() e estão na lista de colunas permitidas 
cat_cols_pipe = [col.name for col in train.schema if (col.dataType ==  StringType() and col.name in cols_to_use)]
ohe_cat_cols = []

for col in cat_cols_pipe:
    indexer = StringIndexer(inputCol=col , outputCol= f'{col}_idx', handleInvalid = 'keep')
    stages.append(indexer)
    one_hot = OneHotEncoder(inputCol=f'{col}_idx' , outputCol= f'{col}_ohe')
    stages.append(one_hot)
    ohe_cat_cols.append(f'{col}_ohe')

In [397]:
# verficiar as colunas restantes
not_cat_cols = list(set(cols_to_use) - set(cat_cols_pipe))
train.select(not_cat_cols).printSchema()

root
 |-- avg_price_per_room: double (nullable = true)
 |-- feriado: integer (nullable = true)
 |-- required_car_parking_space: integer (nullable = true)
 |-- arrival_year: integer (nullable = true)
 |-- repeated_guest: integer (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- no_of_previous_cancellations: integer (nullable = true)
 |-- no_of_week_nights: integer (nullable = true)
 |-- no_of_children: integer (nullable = true)
 |-- duplicated_count: integer (nullable = true)
 |-- arrival_month: integer (nullable = true)
 |-- is_duplicated: boolean (nullable = true)
 |-- no_of_previous_bookings_not_canceled: integer (nullable = true)
 |-- no_of_weekend_nights: integer (nullable = true)
 |-- no_of_special_requests: integer (nullable = true)
 |-- no_of_adults: integer (nullable = true)



In [398]:
# vamos codificá-las tods como númericas
num_proces = VectorAssembler(inputCols = not_cat_cols , outputCol = 'num_features')
stages.append(num_proces)

In [399]:
columns_pipe = ohe_cat_cols + ['num_features']

In [400]:
# juntar tudo em um vetor de features

final_process =  num_proces = VectorAssembler(inputCols = columns_pipe , outputCol = 'features')
stages.append(final_process)

In [401]:
num_cluster = spark.sparkContext.defaultParallelism
print(f"numero de clusters xgb {num_cluster}")

#xgb = SparkXGBClassifier(num_workers= num_cluster, label_col="label", featuresCol = 'features')

numero de clusters xgb 8


In [402]:
xgb_model = SparkXGBClassifier(num_workers= num_cluster, label_col="label", features_col  = 'features')

In [403]:
stages.append(xgb_model)

In [ ]:
pipeline = Pipeline(stages= stages)
pipelineModel = pipeline.fit(train)

2024-12-08 21:56:44,725 INFO XGBoost-PySpark: _fit Running xgboost-2.1.1 on 8 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}


## Otimização de hiperparâmetros

In [362]:
 
paramGrid = ParamGridBuilder()\
  .addGrid(xgb.max_depth, [3, 5,7,9,11,13])\
  .addGrid(xgb.n_estimators, [100, 300,500])\
  .addGrid(xgb.learning_rate, [0.0001, 0.001, 0.01, 0.1, 0.2, 0.3])\
  .build()



In [363]:
# evaluator = BinaryClassificationEvaluator(metricName="areaUnderROC",
#                                 labelCol=xgb.getLabelCol(),
#                                 rawPredictionCol='probability')




In [364]:
cv = CrossValidator(estimator=xgb, estimatorParamMaps=paramGrid, evaluator=BinaryClassificationEvaluator(), numFolds=5)
stages.append(cv)

In [365]:
pipeline = Pipeline(stages= stages)
pipelineModel = pipeline.fit(train)

In [366]:
pipelineModel = pipeline.fit(train)

2024-12-08 21:49:40,548 INFO XGBoost-PySpark: _fit Running xgboost-2.1.1 on 8 workers with
	booster params: {'device': 'cpu', 'max_depth': 2, 'objective': 'binary:logistic', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 10}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}


KeyboardInterrupt: 

2024-12-08 21:52:58,722 INFO XGBoost-PySpark: _fit Running xgboost-2.1.1 on 8 workers with
	booster params: {'device': 'cpu', 'max_depth': 2, 'objective': 'binary:logistic', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2024-12-08 21:57:00,341 INFO XGBoost-PySpark: _fit Running xgboost-2.1.1 on 8 workers with
	booster params: {'device': 'cpu', 'max_depth': 5, 'objective': 'binary:logistic', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 10}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
